In [1]:
import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.utils.data import DataLoader
import importlib
import rasterio

from MachineLearning.predict_testArea_2022 import FOREST_MASK_TIF

_training = importlib.import_module("30_Training")
QuarterBandsDataset = _training.QuarterBandsDataset
MyGRU = _training.MyGRU



In [13]:
VALID_PATH = "/home/main/repositories/RemoteSensing/MachineLearning/dataset/valid.csv"
SPECTRAIL_INVENTORY = "/home/main/repositories/RemoteSensing/Qarterly_mosaics_30m_VelkaFatra/inventory.csv"
FOREST_MASK_PATH = "/home/main/repositories/RemoteSensing/MachineLearning/dataset/cropped_forest_mask_slovakia.tif"

MODEL_PATH = "/home/main/repositories/RemoteSensing/MachineLearning/Model_20261110225617/BestModel.pt"


In [ ]:

dsValid = QuarterBandsDataset(SPECTRAIL_INVENTORY, VALID_PATH)
dataloaderValid = DataLoader(dsValid, batch_size=64, shuffle=False)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [3]:
loss_fn = nn.BCEWithLogitsLoss()

#load_model
checkpoint = torch.load(MODEL_PATH, map_location='cpu', weights_only=False)
state = checkpoint['model_state_dict']

model = MyGRU(loss_fn)
model.load_state_dict(state)
model.to(device)
model.eval()

NameError: name 'MODEL_PATH' is not defined

In [17]:
with rasterio.open(FOREST_MASK_PATH, 'r') as src:
    forest_mask=src.read(1)
    profile = src.profile.copy()

In [4]:
inventory = pd.read_csv(SPECTRAIL_INVENTORY, sep=";")

In [5]:
years = sorted(inventory["Year"].unique())
bands = sorted(inventory["Band"].unique())
y_year = years[2:-1]

inventory = inventory.set_index(["Year","Band"]).sort_index()

In [2]:
def get_x_for_year(year, inve):
    ret=np.empty((4,4,2500,2500))#[y,b,x,y]

    for i, y in enumerate(range(year-2, year+2)):
        for ii, b in enumerate(bands):
            with rasterio.open(inve.loc[(y,b)]["Path"]) as src:
                ret[i,ii] = src.read(1)
    return ret



In [28]:
nan_val = profile['nodata']

for y in y_year:
    x = get_x_for_year(y, inventory)
    prob_for_year = np.zeros(x.shape[2:])
    disturbance_for_year = np.zeros(x.shape[2:])

    for posX in range(0,x.shape[2]):
        print("*", end="")
        for posY in range(0,x.shape[3]):
            if(forest_mask[posX, posY] != 1):
                prob_for_year[posY, posX] = nan_val
                disturbance_for_year[posX, posY] = nan_val
            else:
                mod_point_in = torch.tensor((x[:,:, posX,posY]/1000), dtype=torch.float32).to(device)
                with torch.no_grad():
                    sigm = torch.sigmoid( model(mod_point_in) )
                prob_for_year[posX, posY] = sigm
                disturbance_for_year[posX, posY] = (sigm>0.5).int()

    path_p = f"/home/main/repositories/RemoteSensing/MachineLearning/Model_20261110225617/Valid_predictions/prob_{y}.tiff"
    with rasterio.open(path_p, "w", **profile) as dest:
        dest.write(prob_for_year, 1)

    print(f"saved {path_p}")

    path_d = f"/home/main/repositories/RemoteSensing/MachineLearning/Model_20261110225617/Valid_predictions/disturbance0.5_{y}.tiff"
    with rasterio.open(path_d, "w", **profile) as dest:
        dest.write(disturbance_for_year, 1)

    print(f"saved {path_d}")

****************************************************************************************************************************************************************************************************************************************************************************************************************************************************************************************************************************************************************************************************************************************************************************************************************************************************************************************************************************************************************************************************************************************************************************************************************************************************************************************************************************************

KeyboardInterrupt: 

## Find best treshold

In [39]:
import numpy as np
import torch

change = 0.01
current_treshold = 0
attempts = 100

logits_list = []
trues_list = []

with torch.no_grad():
    for x, y in dataloaderValid:
        # Aplikujeme sigmoid priamo na logity a presunieme na CPU
        probs = torch.sigmoid(model(x.to(device))).cpu()
        logits_list.append(probs)
        trues_list.append(y.cpu())


all_probs = torch.cat(logits_list, dim=0)
all_trues = torch.cat(trues_list, dim=0).view_as(all_probs)

best_acc = 0.0
best_tresh = current_treshold

for i in range(attempts):
    pred = (all_probs > current_treshold).int()

    correct = pred.eq(all_trues.int()).sum().item()
    acc = correct / len(all_probs)

    print(f"tresh: {current_treshold:.4f}, acc: {acc:.4f}")
    row = pd.DataFrame([{"Threshold": current_treshold, "Accuracy": acc}])
    row.to_csv("/home/main/repositories/RemoteSensing/MachineLearning/Model_20261110225617/treshold_vs_acc.csv", mode='a' if i > 0 else 'w',
                   header=(i == 0), index=False)

    if acc > best_acc:
        best_acc = acc
        best_tresh = current_treshold

    current_treshold += change

print(f"\nNajlepšia presnosť: {best_acc:.4f} pri prahu: {best_tresh:.4f}")


tresh: 0.0000, acc: 0.5000
tresh: 0.0100, acc: 0.5843
tresh: 0.0200, acc: 0.6407
tresh: 0.0300, acc: 0.6807
tresh: 0.0400, acc: 0.7097
tresh: 0.0500, acc: 0.7345
tresh: 0.0600, acc: 0.7538
tresh: 0.0700, acc: 0.7700
tresh: 0.0800, acc: 0.7837
tresh: 0.0900, acc: 0.7953
tresh: 0.1000, acc: 0.8058
tresh: 0.1100, acc: 0.8152
tresh: 0.1200, acc: 0.8229
tresh: 0.1300, acc: 0.8301
tresh: 0.1400, acc: 0.8362
tresh: 0.1500, acc: 0.8423
tresh: 0.1600, acc: 0.8477
tresh: 0.1700, acc: 0.8525
tresh: 0.1800, acc: 0.8565
tresh: 0.1900, acc: 0.8603
tresh: 0.2000, acc: 0.8639
tresh: 0.2100, acc: 0.8670
tresh: 0.2200, acc: 0.8702
tresh: 0.2300, acc: 0.8729
tresh: 0.2400, acc: 0.8756
tresh: 0.2500, acc: 0.8780
tresh: 0.2600, acc: 0.8803
tresh: 0.2700, acc: 0.8820
tresh: 0.2800, acc: 0.8841
tresh: 0.2900, acc: 0.8855
tresh: 0.3000, acc: 0.8870
tresh: 0.3100, acc: 0.8885
tresh: 0.3200, acc: 0.8895
tresh: 0.3300, acc: 0.8902
tresh: 0.3400, acc: 0.8909
tresh: 0.3500, acc: 0.8921
tresh: 0.3600, acc: 0.8927
t

## Test Area

In [ ]:
TEST_TRUTHS_PATH = "/home/main/repositories/RemoteSensing/MachineLearning/dataset/TestArea/cropped_annual_disturbances_1985_2023_slovakia.tif"
TEST_SPECTRAIL_INVENTORY = "/home/main/repositories/RemoteSensing/Qarterly_mosaics_30m_TestArea/inventory.csv"
TEST_FOREST_MASK_PATH = "/home/main/repositories/RemoteSensing/MachineLearning/dataset/TestArea/cropped_forest_mask_slovakia.tif"